# HargaWatch Surabaya — Eksperimen Forecasting & Early Warning (E0–E5)

Notebook ini mendokumentasikan serangkaian eksperimen formal untuk peramalan harga pangan dan sistem deteksi dini (*Early Warning System*) Kota Surabaya:
* **E0 — Forecasting EDA & Data Quality**: Uji Stasioneritas (ADF Test), Analisis ACF/PACF, dan Dekomposisi Tren-Musiman (STL).
* **E1 — Naive Baselines**: Evaluasi model naive (*Last Value* dan *7-Day Moving Average*) dengan validasi *Walk-Forward* (Rolling Origin).
* **E2 — Machine Learning Baseline**: Pelatihan *LightGBM Multi-Market Regressor* untuk komoditas pangan bergejolak.
* **E3 — Studi Ablasi Fitur Eksternal**: Membandingkan performa *Price+Calendar* vs *Price+Calendar+Weather* secara empiris.
* **E4 — Interval Prediksi (Uncertainty)**: *Quantile Regression* ({10}, p_{50}, p_{90}$) untuk mengukur rentang ketidakpastian harga.
* **E5 — Kalibrasi Sistem Early Warning (EWS)**: Validasi matriks skoring risiko (*Normal*, *Waspada*, *Tinggi*) terhadap lonjakan harga historis.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Statsmodels & ML
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose

# Add scripts path
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
sys.path.insert(0, str(BASE_DIR))

from scripts.ml.features import (
    load_raw_datasets, prepare_base_series, build_lag_features,
    build_weather_features, build_supervised_dataset
)
from scripts.ml.models import NaiveLastValueForecaster, NaiveSMAForecaster, LightGBMForecaster
from scripts.ml.backtest import compute_metrics, run_walk_forward_backtest
from scripts.ml.early_warning import compute_early_warning_scores

sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
print('Library berhasil dimuat!')


ModuleNotFoundError: No module named 'seaborn'

## 🔬 E0 — Forecasting-Specific EDA & Time-Series Diagnostics

Sebelum melatih model, kita menguji sifat stokastik dari deret waktu harga pangan di Surabaya:
1. **Uji Augmented Dickey-Fuller (ADF)** untuk menguji stasioneritas.
2. **Plot ACF & PACF** untuk mengidentifikasi lag autoregresif dominan.
3. **Dekomposisi Deret Waktu (Trend, Seasonal, Residual)**.


In [ ]:
# Muat dataset silver
df_harga, df_kal, df_cuaca = load_raw_datasets()

# Ambil seri harian Cabe Rawit Merah (Pasar Keputran vs Pasar Wonokromo)
cabe_keputran = df_harga[(df_harga['komoditas_id'] == 50) & (df_harga['pasar_id'] == 5)].sort_values('tanggal').set_index('tanggal')['harga_imputasi']
cabe_wonokromo = df_harga[(df_harga['komoditas_id'] == 50) & (df_harga['pasar_id'] == 2)].sort_values('tanggal').set_index('tanggal')['harga_imputasi']

# 1. ADF Test
def adf_test(series, title=''):
    result = adfuller(series.dropna(), autolag='AIC')
    print(f'=== Uji ADF: {title} ===')
    print(f'ADF Statistic : {result[0]:.4f}')
    print(f'p-value       : {result[1]:.4e}')
    print('Critical Values:')
    for key, value in result[4].items():
        print(f'   {key}: {value:.4f}')
    if result[1] <= 0.05:
        print('-> Kesimpulan : Stasioner (Tolak H0)')
    else:
        print('-> Kesimpulan : Non-Stasioner (Gagal Tolak H0)')

adf_test(cabe_keputran, 'Cabe Rawit Merah (Pasar Keputran)')
adf_test(cabe_wonokromo, 'Cabe Rawit Merah (Pasar Wonokromo)')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_acf(cabe_keputran, lags=30, ax=axes[0], title='Autocorrelation (ACF) - Cabe Rawit Keputran')
plot_pacf(cabe_keputran, lags=30, ax=axes[1], title='Partial Autocorrelation (PACF) - Cabe Rawit Keputran', method='ywm')
plt.tight_layout()
plt.show()


In [ ]:
# Dekomposisi musiman mingguan (period=7)
decomp = seasonal_decompose(cabe_keputran, model='additive', period=7)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp.observed.plot(ax=axes[0], color='#0F172A', title='Observed (Harga Aktual)')
decomp.trend.plot(ax=axes[1], color='#2563EB', title='Trend (Komponen Jangka Panjang)')
decomp.seasonal.tail(180).plot(ax=axes[2], color='#16A34A', title='Weekly Seasonality (Pola Mingguan - 6 Bulan Terakhir)')
decomp.resid.plot(ax=axes[3], color='#DC2626', title='Residual (Kejutan / Shock Tidak Terduga)')
plt.tight_layout()
plt.show()


## 📊 E1 — Naive Baselines & Walk-Forward Backtesting

Kami menguji dua baseline tanpa pelatihan pada data *out-of-sample* (2025–2026) menggunakan validasi *Rolling-Origin*:
1. **Naive Last Value**: $\hat{y}_{t+h} = y_t$
2. **Naive 7-Day SMA**: $\hat{y}_{t+h} = \frac{1}{7}\sum_{i=0}^6 y_{t-i}$


In [ ]:
# Jalankan backtest pada Cabe Rawit Merah (Horizon 7 Hari)
df_res_7d = run_walk_forward_backtest(
    commodity_name='Cabe Rawit Merah',
    horizon=7,
    test_start_date='2025-01-01',
    window_step_days=14,
    include_weather=True
)


## 🌦 E3 — Studi Ablasi Fitur Cuaca (Empirical Ablation Study)

Menguji secara empiris apakah penambahan variabel cuaca (*rain lag* 7d, 14d, 28d) menurunkan error WAPE dan MAE dibanding model murni riwayat harga + kalender.


In [ ]:
# Model Tanpa Cuaca
print('=== UJI ABLASI: TANPA FITUR CUACA ===')
df_no_weather = run_walk_forward_backtest(
    commodity_name='Cabe Rawit Merah',
    horizon=7,
    test_start_date='2025-01-01',
    window_step_days=14,
    include_weather=False
)


## 📈 E4 — Interval Prediksi & Estimasi Ketidakpastian (p10, p50, p90)

Prediksi nilai tunggal (*point forecast*) tidak cukup untuk kebijakan. Model LightGBM menghasilkan rentang batas bawah ({10}$) dan batas atas ({90}$).


In [ ]:
# Melatih model LightGBM Quantile dan menampilkan visualisasi forecast 14 hari
from scripts.run_forecasting import train_and_forecast_commodity

forecast_cabe = train_and_forecast_commodity(komoditas_id=50, horizon_days=14, include_weather=True)

# Plot untuk Pasar Wonokromo (pasar_id=2)
fc_wonokromo = forecast_cabe[forecast_cabe['pasar_id'] == 2].copy()
fc_wonokromo['tanggal'] = pd.to_datetime(fc_wonokromo['tanggal'])

# Ambil 30 hari data historis terakhir untuk konteks
hist_wonokromo = df_harga[(df_harga['komoditas_id'] == 50) & (df_harga['pasar_id'] == 2)].sort_values('tanggal').tail(30)

plt.figure(figsize=(14, 6))
plt.plot(hist_wonokromo['tanggal'], hist_wonokromo['harga_imputasi'], label='Historis Aktual', color='#0F172A', lw=2)
plt.plot(fc_wonokromo['tanggal'], fc_wonokromo['harga_prediksi'], label='Forecast Median (p50)', color='#2563EB', lw=2.5, ls='--')
plt.fill_between(
    fc_wonokromo['tanggal'],
    fc_wonokromo['batas_bawah'],
    fc_wonokromo['batas_atas'],
    color='#93C5FD',
    alpha=0.4,
    label='Interval Prediksi 80% (p10 - p90)'
)
plt.title('Forecasting Cabe Rawit Merah - Pasar Wonokromo (14 Hari ke Depan)', fontsize=14, fontweight='bold')
plt.xlabel('Tanggal')
plt.ylabel('Harga (Rp/Kg)')
plt.legend()
plt.tight_layout()
plt.show()


## 🚨 E5 — Simulasi Matriks Early Warning System (EWS)

Sistem peringatan dini menghitung 4 sub-skor (0–25) untuk mengklasifikasikan risiko pasar:
* NORMAL (< 40)
* WASPADA (40–69)
* TINGGI (>= 70)


In [ ]:
df_base = prepare_base_series(df_harga)
df_lags = build_lag_features(df_base)
latest_date = df_lags['tanggal'].max()

slice_now = df_lags[(df_lags['tanggal'] == latest_date) & (df_lags['komoditas_id'] == 50)]
fc_h7 = forecast_cabe[forecast_cabe['tanggal'] == (latest_date + pd.Timedelta(days=7)).strftime('%Y-%m-%d')]

ews_cabe = compute_early_warning_scores(slice_now, fc_h7)
dim_pasar = pd.read_csv(BASE_DIR / 'data' / 'processed' / 'dim_pasar.csv')

ews_summary = ews_cabe.merge(dim_pasar[['pasar_id', 'nama_pasar']], on='pasar_id')
cols = ['nama_pasar', 'total_skor', 'status_warning', 'skor_tren', 'skor_volatilitas', 'skor_anomali', 'skor_prediksi']
print('=== STATUS EARLY WARNING CABE RAWIT MERAH HARI INI ===')
display(ews_summary[cols])


## 📝 Menampilkan Variabel/Fitur Training

Berikut adalah daftar variabel yang dikonstruksi dan digunakan sebagai input (fitur) ke dalam model LightGBM.

In [ ]:
df_train = build_supervised_dataset(horizon=7, komoditas_id=50, include_weather=True)

print("=== DAFTAR VARIABEL (FITUR) TRAINING ===")
print(f"Total Baris: {len(df_train):,}")
print(f"Total Kolom: {len(df_train.columns)}\n")

features = df_train.columns.tolist()
for i, col in enumerate(features, 1):
    print(f"{i:02d}. {col}")
